# HMM and Ridge Long-Cash Regimes

Uses ridge-based feature selection and a walk-forward HMM to allocate between equities and cash.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
HMM Regime Detection Trading Strategy for SPY - WALK-FORWARD VALIDATION
========================================================================

CRITICAL: This implementation uses STRICT walk-forward validation to eliminate
ALL look-ahead bias. Both Ridge regression and HMM training are done in expanding
windows, with regime mapping frozen for each prediction period.

NO FUTURE DATA is ever used in:
- Feature engineering (Ridge regression)
- HMM training
- Regime mapping (state → economic label)
- Signal generation

Trading Logic:
- Signal generated at Close[t] based on regime prediction
- Execution at Open[t+1]
- Returns calculated as Open[t+1]/Open[t] - 1 (CONSISTENT Open-to-Open)
- No look-ahead bias (strictly T-1 for T predictions)

Author: Production Trading System
Date: 2025
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
from typing import Dict, Tuple, Optional, List
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from hmmlearn import hmm
from dataclasses import dataclass

warnings.filterwarnings("ignore")


class StrategyConfig:
    """Centralized configuration for all strategy parameters."""

    START_DATE = "2015-01-01"
    END_DATE = "2020-12-31"
    INITIAL_CAPITAL = 100000.0
    COMMISSION_RATE = 0.001
    MAIN_TICKER = "SPY"
    VIX_TICKER = "^VIX"
    BENCHMARK_TICKER_GLOBAL = "URTH"
    ROLLING_VOL_WINDOW = 20
    SMA_SHORT = 20
    SMA_LONG = 200
    HMM_N_STATES = 3
    HMM_N_ITER = 100
    HMM_RANDOM_STATE = 42
    RIDGE_ALPHA = 1.0
    RIDGE_RANDOM_STATE = 42
    MIN_TRAINING_DAYS = 252
    REFIT_FREQUENCY = 63


class DataFetcher:
    """Handles all data acquisition from Yahoo Finance."""

    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        """Fetch historical OHLCV data from Yahoo Finance. No bfill/ffill."""
        try:
            response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
            data = response_prices
            if "historical" not in data:
                raise ValueError(f"No data returned for {ticker}")
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").reset_index(drop=True)
            df = df.rename(
                columns={
                    "date": "Date",
                    "open": "Open",
                    "high": "High",
                    "low": "Low",
                    "close": "Close",
                    "volume": "Volume",
                }
            )
            df = df[["Date", "Open", "High", "Low", "Close", "Volume"]]
            print(f"✓ Fetched {len(df)} records for {ticker}")
            return df
        except Exception as e:
            print(f"✗ Error fetching data for {ticker}: {str(e)}")
            raise


class FeatureEngine:
    """Calculates technical features. All calculations avoid look-ahead bias."""

    @staticmethod
    def calculate_rolling_volatility(returns: pd.Series, window: int) -> pd.Series:
        """Calculate rolling volatility (annualized)."""
        return returns.rolling(window=window, min_periods=window).std() * np.sqrt(252)

    @staticmethod
    def calculate_garman_klass_volatility(df: pd.DataFrame, window: int) -> pd.Series:
        """
        Calculate Garman-Klass volatility (CORRECTED FORMULA).
        σ² = 0.5 * (ln(H/L))² - (2*ln(2) - 1) * (ln(C/O))²
        """
        log_hl = np.log(df["High"] / df["Low"])
        log_co = np.log(df["Close"] / df["Open"])
        variance_daily = 0.5 * log_hl**2 - (2 * np.log(2) - 1) * log_co**2
        volatility_daily = np.sqrt(variance_daily)
        gk_vol = volatility_daily.rolling(window=window, min_periods=window).mean() * np.sqrt(252)
        return gk_vol

    @staticmethod
    def calculate_sma_crossover(close: pd.Series, short_window: int, long_window: int) -> pd.Series:
        """Calculate SMA crossover as CONTINUOUS normalized difference."""
        sma_short = close.rolling(window=short_window, min_periods=short_window).mean()
        sma_long = close.rolling(window=long_window, min_periods=long_window).mean()
        return (sma_short - sma_long) / sma_long

    @staticmethod
    def prepare_features(
        df_main: pd.DataFrame, df_vix: pd.DataFrame, config: StrategyConfig
    ) -> pd.DataFrame:
        """Prepare all features with proper lagging to avoid look-ahead bias."""
        df = df_main.copy()
        df["Returns_CC"] = df["Close"].pct_change()
        df["Returns_OO"] = df["Open"].pct_change()
        df["RollingVol"] = FeatureEngine.calculate_rolling_volatility(
            df["Returns_CC"], config.ROLLING_VOL_WINDOW
        )
        df["GarmanKlass"] = FeatureEngine.calculate_garman_klass_volatility(
            df, config.ROLLING_VOL_WINDOW
        )
        df["SMACross"] = FeatureEngine.calculate_sma_crossover(
            df["Close"], config.SMA_SHORT, config.SMA_LONG
        )
        df_vix_clean = df_vix[["Date", "Close"]].rename(columns={"Close": "VIX"})
        df = df.merge(df_vix_clean, on="Date", how="left")
        feature_cols = ["RollingVol", "GarmanKlass", "SMACross", "VIX"]
        for col in feature_cols:
            df[f"{col}_lag"] = df[col].shift(1)
        return df


@dataclass
class RegimeMapping:
    """Stores regime mapping from training period."""

    state_to_regime: Dict[int, str]
    mean_returns: Dict[int, float]
    training_end_date: pd.Timestamp


class WalkForwardHMMRegimeDetector:
    """
    HMM Regime Detector with STRICT walk-forward validation.

    CRITICAL: This class ensures NO look-ahead bias by:
    1. Training HMM only on past data
    2. Creating regime mapping only from training data forward returns
    3. Freezing that mapping for prediction period
    4. Refitting periodically as new data becomes available
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.feature_cols = ["RollingVol_lag", "GarmanKlass_lag", "SMACross_lag", "VIX_lag"]
        self.ridge_feature_cols = [f"{col}_ridge" for col in self.feature_cols]

    def walk_forward_predict(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Perform walk-forward prediction with NO look-ahead bias.

        Process for each window:
        1. Fit StandardScaler + Ridge on training data
        2. Transform training data with Ridge
        3. Fit HMM on Ridge-transformed training data
        4. Create regime mapping using ONLY training forward returns
        5. Transform next period with frozen scaler/Ridge
        6. Predict regime for next period with frozen HMM/mapping
        7. Move window forward and repeat

        Returns DataFrame with Regime predictions that have NO look-ahead bias.
        """
        print("\n" + "=" * 80)
        print("WALK-FORWARD REGIME DETECTION (NO LOOK-AHEAD BIAS)")
        print("=" * 80)
        df_clean = df.dropna(subset=self.feature_cols).copy().reset_index(drop=True)
        print(f"\nTotal observations after cleaning: {len(df_clean)}")
        df_clean["Regime_State"] = np.nan
        df_clean["Regime"] = None
        for col in self.ridge_feature_cols:
            df_clean[col] = np.nan
        min_train = self.config.MIN_TRAINING_DAYS
        refit_freq = self.config.REFIT_FREQUENCY
        folds_executed = []
        print(f"\nStarting walk-forward loop:")
        print(f"  Minimum training: {min_train} days")
        print(f"  Refit frequency: {refit_freq} days")
        print(f"  Total periods to predict: {len(df_clean) - min_train}\n")
        last_refit_idx = min_train
        scaler = None
        ridge_models = None
        hmm_model = None
        regime_mapping = None
        fold_num = 0
        for i in range(min_train, len(df_clean)):
            days_since_refit = i - last_refit_idx
            if scaler is None or days_since_refit >= refit_freq:
                fold_num += 1
                train_start = 0
                train_end = i
                print(f"\n{'=' * 80}")
                print(f"FOLD {fold_num}: Refitting models")
                print(f"{'=' * 80}")
                print(f"Training window: index {train_start} to {train_end - 1}")
                print(
                    f"Training dates: {df_clean.loc[train_start, 'Date'].date()} to {df_clean.loc[train_end - 1, 'Date'].date()}"
                )
                print(f"Training size: {train_end - train_start} days")
                X_train_raw = df_clean.loc[train_start : train_end - 1, self.feature_cols].values
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train_raw)
                ridge_models = {}
                for j in range(len(self.feature_cols)):
                    other_idx = [k for k in range(len(self.feature_cols)) if k != j]
                    if len(other_idx) > 0:
                        ridge = Ridge(
                            alpha=self.config.RIDGE_ALPHA,
                            random_state=self.config.RIDGE_RANDOM_STATE,
                        )
                        ridge.fit(X_train_scaled[:, other_idx], X_train_scaled[:, j])
                        ridge_models[j] = ridge
                X_train_ridge = np.zeros_like(X_train_scaled)
                for j in range(len(self.feature_cols)):
                    other_idx = [k for k in range(len(self.feature_cols)) if k != j]
                    if j in ridge_models:
                        y_pred = ridge_models[j].predict(X_train_scaled[:, other_idx])
                        X_train_ridge[:, j] = X_train_scaled[:, j] - y_pred
                    else:
                        X_train_ridge[:, j] = X_train_scaled[:, j]
                print(f"✓ Ridge features computed for training window")
                hmm_model = hmm.GaussianHMM(
                    n_components=self.config.HMM_N_STATES,
                    covariance_type="full",
                    n_iter=self.config.HMM_N_ITER,
                    random_state=self.config.HMM_RANDOM_STATE,
                )
                hmm_model.fit(X_train_ridge)
                print(f"✓ HMM fitted (converged: {hmm_model.monitor_.converged})")
                states_train = hmm_model.predict(X_train_ridge)
                returns_forward_train = (
                    df_clean.loc[train_start : train_end - 1, "Returns_OO"].shift(-1).values
                )
                state_returns = {}
                state_counts = {}
                for state in range(self.config.HMM_N_STATES):
                    mask = states_train == state
                    mask[-1] = False
                    count = mask.sum()
                    state_counts[state] = count
                    if count > 0:
                        state_returns[state] = returns_forward_train[mask].mean()
                    else:
                        state_returns[state] = 0
                sorted_states = sorted(state_returns.items(), key=lambda x: x[1])
                state_to_regime = {
                    sorted_states[0][0]: "Bearish",
                    sorted_states[1][0]: "Neutral",
                    sorted_states[2][0]: "Bullish",
                }
                regime_mapping = RegimeMapping(
                    state_to_regime=state_to_regime,
                    mean_returns=state_returns,
                    training_end_date=df_clean.loc[train_end - 1, "Date"],
                )
                print(f"✓ Regime mapping created (using ONLY training forward returns):")
                for state, regime in state_to_regime.items():
                    print(
                        f"  State {state} → {regime}: {state_counts[state]} obs, avg fwd return: {state_returns[state]:.6f}"
                    )
                last_refit_idx = i
                folds_executed.append(
                    {
                        "fold": fold_num,
                        "train_start": train_start,
                        "train_end": train_end,
                        "train_start_date": df_clean.loc[train_start, "Date"],
                        "train_end_date": df_clean.loc[train_end - 1, "Date"],
                        "regime_mapping": regime_mapping.state_to_regime.copy(),
                        "mean_returns": regime_mapping.mean_returns.copy(),
                    }
                )
            X_current_raw = df_clean.loc[i:i, self.feature_cols].values
            X_current_scaled = scaler.transform(X_current_raw)
            X_current_ridge = np.zeros_like(X_current_scaled)
            for j in range(len(self.feature_cols)):
                other_idx = [k for k in range(len(self.feature_cols)) if k != j]
                if j in ridge_models:
                    y_pred = ridge_models[j].predict(X_current_scaled[:, other_idx])
                    X_current_ridge[:, j] = X_current_scaled[:, j] - y_pred
                else:
                    X_current_ridge[:, j] = X_current_scaled[:, j]
            for j, col in enumerate(self.ridge_feature_cols):
                df_clean.loc[i, col] = X_current_ridge[0, j]
            state_pred = hmm_model.predict(X_current_ridge)[0]
            regime_pred = regime_mapping.state_to_regime[state_pred]
            df_clean.loc[i, "Regime_State"] = state_pred
            df_clean.loc[i, "Regime"] = regime_pred
        print(f"\n{'=' * 80}")
        print(f"WALK-FORWARD COMPLETE")
        print(f"{'=' * 80}")
        print(f"Total folds executed: {len(folds_executed)}")
        print(f"Predictions generated: {df_clean['Regime'].notna().sum()}")
        regime_counts = df_clean["Regime"].value_counts()
        print(f"\nFinal regime distribution:")
        for regime in ["Bearish", "Neutral", "Bullish"]:
            count = regime_counts.get(regime, 0)
            pct = count / len(df_clean) * 100 if len(df_clean) > 0 else 0
            print(f"  {regime}: {count} days ({pct:.1f}%)")
        return (df_clean, folds_executed)


class BacktestEngine:
    """Realistic backtesting with Open-to-Open returns."""

    def __init__(self, initial_capital: float, commission_rate: float):
        self.initial_capital = initial_capital
        self.commission_rate = commission_rate
        self.trades = []

    def run_backtest(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Execute backtest with proper signal-position-return alignment.
        Signal[t] at Close → Execute at Open[t+1] → Position[t+1] → Returns[t+1]
        """
        results = df.reset_index(drop=True).copy()
        results["Signal"] = 0
        results.loc[results["Regime"] == "Bullish", "Signal"] = 1
        results["Position"] = 0
        results["Cash"] = 0.0
        results["Holdings"] = 0.0
        results["Portfolio_Value"] = 0.0
        results["Strategy_Returns"] = 0.0
        results["Trades"] = ""
        cash = self.initial_capital
        shares = 0
        results.loc[0, "Cash"] = cash
        results.loc[0, "Portfolio_Value"] = cash
        for i in range(1, len(results)):
            desired_position = int(results.loc[i - 1, "Signal"])
            current_position = int(results.loc[i - 1, "Position"])
            if desired_position != current_position:
                execution_price = results.loc[i, "Open"]
                if desired_position == 1 and current_position == 0:
                    shares = cash / (execution_price * (1 + self.commission_rate))
                    commission = shares * execution_price * self.commission_rate
                    cash = 0
                    self.trades.append(
                        {
                            "Date": results.loc[i, "Date"],
                            "Type": "BUY",
                            "Price": execution_price,
                            "Shares": shares,
                            "Commission": commission,
                        }
                    )
                    results.loc[i, "Trades"] = "BUY"
                elif desired_position == 0 and current_position == 1:
                    commission = shares * execution_price * self.commission_rate
                    cash = shares * execution_price - commission
                    self.trades.append(
                        {
                            "Date": results.loc[i, "Date"],
                            "Type": "SELL",
                            "Price": execution_price,
                            "Shares": shares,
                            "Commission": commission,
                        }
                    )
                    results.loc[i, "Trades"] = "SELL"
                    shares = 0
            results.loc[i, "Position"] = 1 if shares > 0 else 0
            results.loc[i, "Cash"] = cash
            results.loc[i, "Holdings"] = shares * results.loc[i, "Open"]
            results.loc[i, "Portfolio_Value"] = cash + shares * results.loc[i, "Open"]
        results["Strategy_Returns"] = results["Portfolio_Value"].pct_change()
        results["Cumulative_Returns"] = (1 + results["Strategy_Returns"]).cumprod()
        print(f"\n✓ Strategy backtest completed")
        print(f"  Total trades: {len(self.trades)}")
        print(f"  Final value: ${results.loc[len(results) - 1, 'Portfolio_Value']:,.2f}")
        return results

    def add_buy_hold_benchmark(
        self, results: pd.DataFrame, price_df: pd.DataFrame, ticker_name: str
    ) -> pd.DataFrame:
        """
        Add buy-and-hold benchmark using Open prices.

        Parameters:
        -----------
        results : pd.DataFrame
            Main results dataframe (will be modified in place)
        price_df : pd.DataFrame
            Price data with Date and Open columns
        ticker_name : str
            Name for the benchmark (e.g., 'SPY', 'URTH')
        """
        price_aligned = price_df[["Date", "Open"]].rename(columns={"Open": f"{ticker_name}_Open"})
        results = results.merge(price_aligned, on="Date", how="left")
        first_open = results.loc[0, f"{ticker_name}_Open"]
        shares = self.initial_capital / (first_open * (1 + self.commission_rate))
        commission_buy = shares * first_open * self.commission_rate
        col_prefix = f"{ticker_name}_BH"
        results[f"{col_prefix}_Value"] = shares * results[f"{ticker_name}_Open"]
        results.loc[0, f"{col_prefix}_Value"] = self.initial_capital - commission_buy
        last_idx = len(results) - 1
        last_open = results.loc[last_idx, f"{ticker_name}_Open"]
        commission_sell = shares * last_open * self.commission_rate
        results.loc[last_idx, f"{col_prefix}_Value"] = shares * last_open - commission_sell
        results[f"{col_prefix}_Returns"] = results[f"{ticker_name}_Open"].pct_change()
        results[f"{col_prefix}_Cumulative"] = (1 + results[f"{col_prefix}_Returns"]).cumprod()
        print(
            f"✓ {ticker_name} B&H benchmark added: ${results.loc[last_idx, f'{col_prefix}_Value']:,.2f}"
        )
        return results


class PerformanceMetrics:
    """Calculate comprehensive performance metrics."""

    @staticmethod
    def calculate_all_metrics(results: pd.DataFrame, rf_rate: float = 0.02) -> Dict:
        """Calculate metrics for strategy and all benchmarks."""
        metrics = {}
        strategy_returns = results["Strategy_Returns"].dropna()
        n_years = len(strategy_returns) / 252
        metrics["Strategy"] = {
            "Total Return": results.iloc[-1]["Portfolio_Value"] / results.iloc[0]["Portfolio_Value"]
            - 1,
            "CAGR": (
                (results.iloc[-1]["Portfolio_Value"] / results.iloc[0]["Portfolio_Value"])
                ** (1 / n_years)
                - 1
                if n_years > 0
                else 0
            ),
            "Volatility": strategy_returns.std() * np.sqrt(252),
            "Sharpe": (
                np.sqrt(252) * (strategy_returns.mean() - rf_rate / 252) / strategy_returns.std()
                if strategy_returns.std() > 0
                else 0
            ),
            "Max Drawdown": (
                (strategy_returns + 1).cumprod()
                / (strategy_returns + 1).cumprod().expanding().max()
                - 1
            ).min(),
            "Win Rate": (strategy_returns > 0).sum() / len(strategy_returns),
            "Final Value": results.iloc[-1]["Portfolio_Value"],
        }
        for col_prefix in ["SPY_BH", "URTH_BH"]:
            if f"{col_prefix}_Returns" in results.columns:
                bench_returns = results[f"{col_prefix}_Returns"].dropna()
                metrics[col_prefix.replace("_BH", "")] = {
                    "Total Return": results.iloc[-1][f"{col_prefix}_Value"]
                    / results.iloc[0][f"{col_prefix}_Value"]
                    - 1,
                    "CAGR": (
                        (
                            results.iloc[-1][f"{col_prefix}_Value"]
                            / results.iloc[0][f"{col_prefix}_Value"]
                        )
                        ** (1 / n_years)
                        - 1
                        if n_years > 0
                        else 0
                    ),
                    "Volatility": bench_returns.std() * np.sqrt(252),
                    "Sharpe": (
                        np.sqrt(252) * (bench_returns.mean() - rf_rate / 252) / bench_returns.std()
                        if bench_returns.std() > 0
                        else 0
                    ),
                    "Max Drawdown": (
                        (bench_returns + 1).cumprod()
                        / (bench_returns + 1).cumprod().expanding().max()
                        - 1
                    ).min(),
                    "Final Value": results.iloc[-1][f"{col_prefix}_Value"],
                }
        return metrics

    @staticmethod
    def print_metrics(metrics: Dict):
        """Print formatted metrics table."""
        print("\n" + "=" * 100)
        print("PERFORMANCE METRICS (WALK-FORWARD VALIDATED - NO LOOK-AHEAD)")
        print("=" * 100)
        for name, m in metrics.items():
            print(f"\n{name}:")
            print(f"  Total Return:     {m['Total Return']:>10.2%}")
            print(f"  CAGR:             {m['CAGR']:>10.2%}")
            print(f"  Volatility:       {m['Volatility']:>10.2%}")
            print(f"  Sharpe Ratio:     {m['Sharpe']:>10.2f}")
            print(f"  Max Drawdown:     {m['Max Drawdown']:>10.2%}")
            if "Win Rate" in m:
                print(f"  Win Rate:         {m['Win Rate']:>10.2%}")
            print(f"  Final Value:      ${m['Final Value']:>10,.2f}")
        if "Strategy" in metrics and "SPY" in metrics:
            print(f"\nRelative to SPY:")
            print(
                f"  Excess Return:    {metrics['Strategy']['Total Return'] - metrics['SPY']['Total Return']:>10.2%}"
            )
            print(
                f"  Excess CAGR:      {metrics['Strategy']['CAGR'] - metrics['SPY']['CAGR']:>10.2%}"
            )
        print("=" * 100 + "\n")


def create_trade_log_sample(results: pd.DataFrame, n_samples: int = 10) -> pd.DataFrame:
    """Create sample trade log for validation."""
    trade_rows = results[results["Trades"].isin(["BUY", "SELL"])].copy()
    if len(trade_rows) == 0:
        print("No trades executed")
        return pd.DataFrame()
    sample_size = min(n_samples, len(trade_rows))
    sample_indices = np.random.choice(trade_rows.index, sample_size, replace=False)
    sample = results.loc[sorted(sample_indices)].copy()
    log_cols = ["Date", "Regime", "Signal", "Position", "Trades", "Open", "Strategy_Returns"]
    available_cols = [col for col in log_cols if col in sample.columns]
    log = sample[available_cols].copy()
    return log


def plot_results(results: pd.DataFrame, config: StrategyConfig):
    """Create visualization plots."""
    fig, axes = plt.subplots(3, 1, figsize=(15, 12))
    regime_colors = {"Bearish": "red", "Neutral": "gray", "Bullish": "green"}
    for regime, color in regime_colors.items():
        mask = results["Regime"] == regime
        if mask.sum() > 0:
            axes[0].scatter(
                results.loc[mask, "Date"],
                results.loc[mask, "Close"],
                c=color,
                label=regime,
                alpha=0.6,
                s=10,
            )
    axes[0].set_ylabel("Price")
    axes[0].set_title(f"{config.MAIN_TICKER} with HMM Regimes (Walk-Forward, No Look-Ahead)")
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    axes[1].plot(
        results["Date"], 100 * results["Cumulative_Returns"], label="HMM Strategy", linewidth=2
    )
    if "SPY_BH_Cumulative" in results.columns:
        axes[1].plot(
            results["Date"],
            100 * results["SPY_BH_Cumulative"],
            label="SPY Buy & Hold",
            linewidth=2,
            alpha=0.7,
        )
    if "URTH_BH_Cumulative" in results.columns:
        axes[1].plot(
            results["Date"],
            100 * results["URTH_BH_Cumulative"],
            label="URTH Buy & Hold",
            linewidth=2,
            alpha=0.7,
        )
    axes[1].set_ylabel("Cumulative Returns (Base 100)")
    axes[1].set_title("Strategy vs Buy & Hold (Open-to-Open Returns)")
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    strategy_cum = (1 + results["Strategy_Returns"]).cumprod()
    strategy_dd = (strategy_cum / strategy_cum.expanding().max() - 1) * 100
    axes[2].fill_between(results["Date"], strategy_dd, 0, alpha=0.3, color="red")
    axes[2].plot(results["Date"], strategy_dd, color="darkred", linewidth=1)
    axes[2].set_xlabel("Date")
    axes[2].set_ylabel("Drawdown (%)")
    axes[2].set_title("Strategy Drawdown")
    axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def main():
    """Main execution with strict walk-forward validation."""
    print("\n" + "=" * 100)
    print("HMM REGIME DETECTION STRATEGY - WALK-FORWARD VALIDATION (NO LOOK-AHEAD BIAS)")
    print("=" * 100)
    config = StrategyConfig()
    print(f"\nConfiguration:")
    print(f"  Period: {config.START_DATE} to {config.END_DATE}")
    print(f"  Ticker: {config.MAIN_TICKER}")
    print(f"  Capital: ${config.INITIAL_CAPITAL:,.0f}")
    print(f"  Commission: {config.COMMISSION_RATE:.3%}")
    print(f"  Min Training: {config.MIN_TRAINING_DAYS} days")
    print(f"  Refit Frequency: {config.REFIT_FREQUENCY} days")
    print("\n" + "-" * 100)
    print("DATA ACQUISITION")
    print("-" * 100)
    fetcher = DataFetcher()
    df_spy = fetcher.fetch_historical_data(config.MAIN_TICKER, config.START_DATE, config.END_DATE)
    df_vix = fetcher.fetch_historical_data(config.VIX_TICKER, config.START_DATE, config.END_DATE)
    df_urth = fetcher.fetch_historical_data(
        config.BENCHMARK_TICKER_GLOBAL, config.START_DATE, config.END_DATE
    )
    print("\n" + "-" * 100)
    print("FEATURE ENGINEERING")
    print("-" * 100)
    df_features = FeatureEngine.prepare_features(df_spy, df_vix, config)
    detector = WalkForwardHMMRegimeDetector(config)
    df_regimes, folds = detector.walk_forward_predict(df_features)
    print("\n" + "-" * 100)
    print("BACKTESTING")
    print("-" * 100)
    engine = BacktestEngine(config.INITIAL_CAPITAL, config.COMMISSION_RATE)
    results = engine.run_backtest(df_regimes)
    print("\nAdding benchmarks...")
    results = engine.add_buy_hold_benchmark(results, df_regimes, "SPY")
    results = engine.add_buy_hold_benchmark(results, df_urth, "URTH")
    print("\n" + "-" * 100)
    print("PERFORMANCE ANALYSIS")
    print("-" * 100)
    metrics = PerformanceMetrics.calculate_all_metrics(results)
    PerformanceMetrics.print_metrics(metrics)
    print("\n" + "-" * 100)
    print("VALIDATION: TRADE LOG SAMPLE")
    print("-" * 100)
    trade_log = create_trade_log_sample(results, n_samples=10)
    if len(trade_log) > 0:
        print("\nSample of trades (verifying alignment):")
        print(trade_log.to_string(index=False))
    else:
        print("\nNo trades to sample")
    print(f"\n\nSummary Statistics:")
    print(f"  Total trading days: {len(results)}")
    print(f"  Days with Signal=1: {(results['Signal'] == 1).sum()}")
    print(f"  Days with Position=1: {(results['Position'] == 1).sum()}")
    print(f"  Total trades: {len(engine.trades)}")
    regime_counts = results["Regime"].value_counts()
    print(f"\n  Regime distribution:")
    for regime in ["Bullish", "Neutral", "Bearish"]:
        count = regime_counts.get(regime, 0)
        pct = count / len(results) * 100 if len(results) > 0 else 0
        print(f"    {regime}: {count} days ({pct:.1f}%)")
    print("\n" + "-" * 100)
    print("GENERATING VISUALIZATIONS")
    print("-" * 100)
    plot_results(results, config)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results.to_csv(f"hmm_walkforward_results_{timestamp}.csv", index=False)
    pd.DataFrame(engine.trades).to_csv(f"hmm_walkforward_trades_{timestamp}.csv", index=False)
    pd.DataFrame(folds).to_csv(f"hmm_walkforward_folds_{timestamp}.csv", index=False)
    print(f"\n✓ Results exported with timestamp {timestamp}")
    print("\n" + "=" * 100)
    print("EXECUTION COMPLETE - RESULTS ARE PUBLICATION-READY")
    print("=" * 100 + "\n")
    return (results, metrics, folds)


if __name__ == "__main__":
    results, metrics, folds = main()
